# Competition Analysis — AI Actors Network

Analyse concurrentielle à partir des données de la base SQLite :
1. Export des couples entreprise / concurrent
2. Format long + agrégation
3. Matrice de cooccurrence
4. Projection 2D

## 1. Configuration et imports

In [25]:
import sqlite3
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.preprocessing import normalize

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

COL_NAME        = "name"
COL_SECTOR      = "sector"
COL_COMPETITORS = "main_competitors"
RANDOM_SEED     = 42

print(f"Base : {DB_PATH}")
print(f"Exports : {EXPORTS_DIR}")

Base : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Exports : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports


## 2. Extraction SQL — entreprise / secteurs / concurrents

In [26]:
TOP_N = 100   # nombre d'entreprises retenues pour l'analyse

# ranking_score = MAX(capitalization, funds_raised) en numérique — même logique que server.js
SQL = f"""
SELECT
    name             AS {COL_NAME},
    sector           AS {COL_SECTOR},
    main_competitors AS {COL_COMPETITORS},
    MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
    ) AS ranking_score
FROM enterprises
WHERE main_competitors IS NOT NULL
  AND main_competitors != ''
  AND MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
      ) > 0
ORDER BY ranking_score DESC
LIMIT {TOP_N}
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(f"{len(df_raw)} entreprises (top {TOP_N} par capitalisation / fonds levés)")
df_raw[["name", "ranking_score"]].head(10)

45 entreprises (top 100 par capitalisation / fonds levés)


,name,ranking_score
0,ByteDance,50000.0
1,Alibaba,30000.0
2,Scale AI,16000.0
3,Wayve,2500.0
4,UiPath,1960.0
5,Celonis,1770.0
6,YouTube,1680.0
7,Cohere,1600.0
8,World Labs,1230.0
9,Harvey,1200.0


In [27]:
# Export brut
raw_path = EXPORTS_DIR / "competitors_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut → {raw_path}")

Export brut → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_raw.csv


## 3. Nettoyage et standardisation des noms de concurrents

In [28]:
_SEP = re.compile(r"[,;/\n]+")
_INVALID = re.compile(r"^(na|n/a|none|unknown|tbd|-)$", re.IGNORECASE)

def clean_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\(.*?\)", "", s).strip()
    s = re.sub(r"\s{2,}", " ", s)
    return s.title()

def split_competitors(raw: str) -> list[str]:
    parts = _SEP.split(str(raw))
    return [clean_name(p) for p in parts
            if clean_name(p) and not _INVALID.match(p.strip())]

df_clean = df_raw.copy()
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(split_competitors)

df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS] if c.lower() != r[COL_NAME].lower()],
    axis=1,
)

df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

print(f"{len(df_clean)} entreprises après nettoyage")
df_clean[[COL_NAME, COL_COMPETITORS]].head(8)

41 entreprises après nettoyage


,name,main_competitors
0,ByteDance,"[Meta, Youtube, Tencent, Openai, Google Deepmind]"
1,Alibaba,"[Pinduoduo, Jd.Com, Bytedance, Temu, Shein, Am..."
2,Scale AI,"[Srge Ai, Snorkel Ai, Data Annotation, Appen, ..."
3,Wayve,[Waymo]
4,Celonis,"[Sap Signavio, Microsoft Power Automate, Uipat..."
5,YouTube,"[Tiktok, Instagram, Twitch, Vimeo, Deezer]"
6,Cohere,"[Anthropic, Mistral, Openai, Meta]"
7,World Labs,"[Ami Labs, Google Deepmind, Odyssey]"


## 3b. Normalisation sémantique — filiales → groupe parent

In [29]:
SEMANTIC_ALIASES: dict[str, str] = {
    # ── Alphabet / Google ─────────────────────────────────────────────────────
    "Google":               "Alphabet",
    "Alphabet Inc.":        "Alphabet",
    "Youtube":              "Alphabet",
    "Deepmind":             "Alphabet",
    "Google Deepmind":      "Alphabet",
    "Google Brain":         "Alphabet",
    "Google Cloud":         "Alphabet",
    "Waymo":                "Alphabet",
    "Verily":               "Alphabet",
    "Calico":               "Alphabet",
    "Waze":                 "Alphabet",
    # ── Meta Platforms ────────────────────────────────────────────────────────
    "Facebook":             "Meta Platforms",
    "Instagram":            "Meta Platforms",
    "Whatsapp":             "Meta Platforms",
    "Threads":              "Meta Platforms",
    "Meta":                 "Meta Platforms",
    # ── Microsoft ─────────────────────────────────────────────────────────────
    "Azure":                "Microsoft",
    "Linkedin":             "Microsoft",
    "Github":               "Microsoft",
    "Skype":                "Microsoft",
    "Bing":                 "Microsoft",
    "Nuance":               "Microsoft",
    "Nuance Communications":"Microsoft",
    "Activision Blizzard":  "Microsoft",
    "Activision":           "Microsoft",
    # ── Amazon ────────────────────────────────────────────────────────────────
    "Aws":                  "Amazon",
    "Amazon Web Services":  "Amazon",
    "Alexa":                "Amazon",
    "Twitch":               "Amazon",
    "Amazon.Com":           "Amazon",
    # ── Apple ─────────────────────────────────────────────────────────────────
    "Siri":                 "Apple",
    "Apple Inc.":           "Apple",
    # ── Salesforce ────────────────────────────────────────────────────────────
    "Slack":                "Salesforce",
    "Tableau":              "Salesforce",
    "Mulesoft":             "Salesforce",
    # ── IBM ───────────────────────────────────────────────────────────────────
    "Red Hat":              "IBM",
    "Watsonx":              "IBM",
    "Watson":               "IBM",
    # ── Oracle ────────────────────────────────────────────────────────────────
    "Netsuite":             "Oracle",
    # ── Nvidia ────────────────────────────────────────────────────────────────
    "Cuda":                 "Nvidia",
    "Nvidia Corporation":   "Nvidia",
    # ── ByteDance ─────────────────────────────────────────────────────────────
    "Tiktok":               "Bytedance",
    "Douyin":               "Bytedance",
    "Bytedance":            "Bytedance",
    # ── X / Twitter ───────────────────────────────────────────────────────────
    "Twitter":              "X",
    "X.Com":                "X",
    # ── Baidu ─────────────────────────────────────────────────────────────────
    "Ernie":                "Baidu",
    "Ernie Bot":            "Baidu",
    # ── Tencent ───────────────────────────────────────────────────────────────
    "Wechat":               "Tencent",
    "Qq":                   "Tencent",
    # ── OpenAI ────────────────────────────────────────────────────────────────
    "Chatgpt":              "OpenAI",
    "Gpt-4":                "OpenAI",
    "Gpt4":                 "OpenAI",
    "Openai":               "OpenAI",
    # ── Samsung ───────────────────────────────────────────────────────────────
    "Samsung Electronics":  "Samsung",
    # ── Adobe ─────────────────────────────────────────────────────────────────
    "Adobe Firefly":        "Adobe",
}

def apply_semantic_aliases(names: list[str]) -> list[str]:
    return [SEMANTIC_ALIASES.get(n, n) for n in names]

df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(apply_semantic_aliases)
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(lambda lst: list(dict.fromkeys(lst)))
df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS]
               if c.lower() != r[COL_NAME].lower()
               and SEMANTIC_ALIASES.get(r[COL_NAME].title(), r[COL_NAME]).lower() != c.lower()],
    axis=1,
)
df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

# explode garde le nom COL_COMPETITORS, pas "competitor"
preview = df_clean[[COL_NAME, COL_COMPETITORS]].explode(COL_COMPETITORS)
print(f"{len(SEMANTIC_ALIASES)} aliases | {len(df_clean)} entreprises après normalisation sémantique")
print("\nTop concurrents après normalisation :")
print(preview[COL_COMPETITORS].value_counts().head(15).to_string())

56 aliases | 41 entreprises après normalisation sémantique

Top concurrents après normalisation :
main_competitors
Alphabet          9
OpenAI            8
Amazon            6
Meta Platforms    5
Microsoft         5
Bytedance         4
Anthropic         4
Intel             4
Apple             4
Tencent           3
Nvidia            3
Qualcomm          3
Broadcom          2
Exscientia        2
Samsung           2


## 4. Format long — couples entreprise–concurrent

In [30]:
df_long = (
    df_clean
    .explode(COL_COMPETITORS)
    .rename(columns={COL_COMPETITORS: "competitor"})
    .reset_index(drop=True)
    [[COL_NAME, "competitor", COL_SECTOR, "ranking_score"]]
)
df_long.insert(0, "pair_id", df_long.index)

long_path = EXPORTS_DIR / "competitors_long.csv"
df_long.to_csv(long_path, index=False)

print(f"{len(df_long)} couples entreprise–concurrent")
print(f"Export → {long_path}")
df_long.head(10)

236 couples entreprise–concurrent
Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_long.csv


,pair_id,name,competitor,sector,ranking_score
0,0,ByteDance,Meta Platforms,Media & Entertainment,50000.0
1,1,ByteDance,Alphabet,Media & Entertainment,50000.0
2,2,ByteDance,Tencent,Media & Entertainment,50000.0
3,3,ByteDance,OpenAI,Media & Entertainment,50000.0
4,4,Alibaba,Pinduoduo,"Cloud Provider, Financial Services, Retail & E...",30000.0
5,5,Alibaba,Jd.Com,"Cloud Provider, Financial Services, Retail & E...",30000.0
6,6,Alibaba,Bytedance,"Cloud Provider, Financial Services, Retail & E...",30000.0
7,7,Alibaba,Temu,"Cloud Provider, Financial Services, Retail & E...",30000.0
8,8,Alibaba,Shein,"Cloud Provider, Financial Services, Retail & E...",30000.0
9,9,Alibaba,Amazon,"Cloud Provider, Financial Services, Retail & E...",30000.0


## 5. Agrégation et comptage des relations concurrentielles

In [31]:
df_agg = (
    df_long
    .groupby([COL_NAME, "competitor"], sort=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 paires entreprise–concurrent :")
display(df_agg.head(20))

print("\nDistribution des fréquences :")
display(df_agg["count"].describe())

agg_path = EXPORTS_DIR / "competitors_aggregated.csv"
df_agg.to_csv(agg_path, index=False)
print(f"\nExport → {agg_path}")

Top 20 paires entreprise–concurrent :


,name,competitor,count
0,ByteDance,Meta Platforms,1
1,ByteDance,Alphabet,1
2,ByteDance,Tencent,1
3,ByteDance,OpenAI,1
4,Alibaba,Pinduoduo,1
5,Alibaba,Jd.Com,1
6,Alibaba,Bytedance,1
7,Alibaba,Temu,1
8,Alibaba,Shein,1
9,Alibaba,Amazon,1



Distribution des fréquences :


count    236.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: count, dtype: float64


Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_aggregated.csv


## 6. Matrice de cooccurrence

In [32]:
# Tableau croisé entreprise × concurrent (asymétrique)
pivot = df_agg.pivot_table(
    index=COL_NAME, columns="competitor", values="count", fill_value=0
)

all_actors = sorted(set(pivot.index) | set(pivot.columns))
pivot = pivot.reindex(index=all_actors, columns=all_actors, fill_value=0)

# Cooccurrence symétrique : M + M^T
cooc = pivot.values + pivot.values.T
np.fill_diagonal(cooc, 0)
df_cooc = pd.DataFrame(cooc, index=all_actors, columns=all_actors)

print(f"Matrice {df_cooc.shape[0]} × {df_cooc.shape[1]} acteurs")
print(f"Densité non-nulle : {(cooc > 0).mean():.1%}")

cooc_path = EXPORTS_DIR / "cooccurrence_matrix.csv"
df_cooc.to_csv(cooc_path)
print(f"Export → {cooc_path}")

Matrice 223 × 223 acteurs
Densité non-nulle : 0.9%
Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_matrix.csv


## 7. Projection 2D de la cooccurrence

On choisit la méthode selon la taille : PCA si > 500 acteurs, MDS sinon (MDS préserve mieux les distances de cooccurrence pour des jeux de taille raisonnable).

In [33]:
import warnings

N = len(all_actors)
X = normalize(cooc, norm="l2")

score_map        = df_agg.groupby("competitor")["count"].sum().to_dict()
ranking_score_map = df_raw.set_index(COL_NAME)["ranking_score"].to_dict()

if N > 500:
    print(f"PCA (N={N} > 500)")
    coords = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X)
    method = "PCA"
else:
    print(f"MDS (N={N} ≤ 500)")
    sim = X @ X.T
    dissim = 1 - np.clip(sim, 0, 1)
    np.fill_diagonal(dissim, 0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        reducer = MDS(n_components=2, dissimilarity="precomputed",
                      init="random", random_state=RANDOM_SEED, normalized_stress="auto")
        coords = reducer.fit_transform(dissim)
    method = "MDS"

df_coords = pd.DataFrame({"actor": all_actors, "x": coords[:, 0], "y": coords[:, 1]})
df_coords["score"]         = df_coords["actor"].map(score_map).fillna(0)
df_coords["log_score"]     = np.log10(df_coords["score"] + 1)
df_coords["ranking_score"] = df_coords["actor"].map(ranking_score_map).fillna(0)

sector_map = (
    df_raw.set_index(COL_NAME)[COL_SECTOR]
    .dropna()
    .apply(lambda s: s.split(",")[0].strip())
    .to_dict()
)
df_coords["sector"] = df_coords["actor"].map(sector_map).fillna("External")

coords_path = EXPORTS_DIR / "coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Coordonnées 2D → {coords_path}")
df_coords.sort_values("ranking_score", ascending=False).head(10)

MDS (N=223 ≤ 500)


Coordonnées 2D → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_2d.csv


,actor,x,y,score,log_score,ranking_score,sector
37,ByteDance,-0.738420,0.191714,0.0,0.0,50000.0,Media & Entertainment
7,Alibaba,-0.719334,-0.284572,0.0,0.0,30000.0,Cloud Provider
168,Scale AI,0.718531,0.271990,0.0,0.0,16000.0,ICT
217,Wayve,-0.712539,0.235431,0.0,0.0,2500.0,ICT
41,Celonis,-0.237866,-0.658950,0.0,0.0,1770.0,ICT
222,YouTube,-0.699062,0.308692,0.0,0.0,1680.0,Media & Entertainment
49,Cohere,-0.750191,-0.200056,0.0,0.0,1600.0,AI model
219,World Labs,-0.575184,-0.238085,0.0,0.0,1230.0,AI model
81,Harvey,0.751183,0.107535,0.0,0.0,1200.0,Professional Services
3,AMD,-0.531086,0.136056,0.0,0.0,794.0,Hardware


## 8. Visualisation 2D et export des artefacts

In [34]:
import plotly.express as px

info_cols = ["founded_year", "country", "employees_count",
             "revenue_millions", "capitalization", "funds_raised", "description"]
available = [c for c in info_cols if c in df_raw.columns]
df_info   = df_raw.set_index(COL_NAME)[available]

df_plot = df_coords.copy()
for col in available:
    df_plot[col] = df_plot["actor"].map(df_info[col])

def fmt_hover(row):
    lines = [f"<b>{row['actor']}</b>"]
    if pd.notna(row.get("sector")) and row["sector"] != "External":
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")):
        lines.append(f"Country: {row['country']}")
    if pd.notna(row.get("founded_year")):
        lines.append(f"Founded: {int(row['founded_year'])}")
    if pd.notna(row.get("employees_count")):
        lines.append(f"Employees: {int(row['employees_count']):,}")
    cap = float(row.get("capitalization") or 0)
    if cap > 0:
        lines.append(f"Market cap: {cap/1000:.1f}B USD")
    rev = float(row.get("revenue_millions") or 0)
    if rev > 0:
        lines.append(f"Revenue: {rev/1000:.1f}B USD")
    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:160].rstrip()
        lines.append(f"<i>{snippet}{'…' if len(desc) > 160 else ''}</i>")
    lines.append(f"Citations: {int(row['score'])}")
    return "<br>".join(lines)

df_plot["hover"]       = df_plot.apply(fmt_hover, axis=1)
df_plot["marker_size"] = np.maximum(df_plot["score"] * 2, 3)

fig = px.scatter(
    df_plot,
    x="x", y="y",
    color="sector",
    size="marker_size",
    size_max=22,
    text="actor",
    custom_data=["hover"],
    color_discrete_sequence=px.colors.qualitative.Light24,
    title=f"Espace concurrentiel — {method}  ({N} acteurs · taille ∝ citations)",
    labels={"x": "Dimension 1", "y": "Dimension 2", "sector": "Secteur"},
    height=800,
)

fig.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(opacity=0.78, line=dict(width=0.4, color="white")),
)

fig.update_layout(
    legend=dict(title="Secteur", font=dict(size=10)),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
)

fig_html = EXPORTS_DIR / "competition_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive → {fig_html}")
fig.show()

Carte interactive → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.html


In [35]:
print("── Récapitulatif des exports ────────────────────────────")
for p in [raw_path, long_path, agg_path, cooc_path, coords_path, fig_html]:
    size_kb = Path(p).stat().st_size / 1024
    print(f"  {p.name:<42} {size_kb:6.1f} KB")

── Récapitulatif des exports ────────────────────────────
  competitors_raw.csv                           4.9 KB
  competitors_long.csv                         12.2 KB
  competitors_aggregated.csv                    5.3 KB
  cooccurrence_matrix.csv                     199.3 KB
  coords_2d.csv                                18.7 KB
  competition_map_2d.html                    4783.6 KB
